# Lab 5: Logowanie i wizualizacja

---

## Zadanie 1: Tensorboard podstawowe operacje

In [1]:
import math
import torch
import torch.nn as nn

from torch.utils.tensorboard import SummaryWriter

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

writer = SummaryWriter('runs/test1')

model = nn.Sequential(
    nn.Linear(1,5),
    nn.Linear(5,5),
    nn.Linear(5,1)
).to(DEVICE)

sample_x = torch.tensor([1.0]).unsqueeze(0)
writer.add_graph(model,sample_x.to(DEVICE))

In [2]:
for trial in range(1000):
    loss = math.exp(-trial/30)
    writer.add_scalars(
        main_tag='optuna',
        tag_scalar_dict={'loss':loss},
        global_step=trial
    )

    writer.flush()

In [3]:
writer = SummaryWriter('runs/test2')
model = nn.Sequential(nn.Linear(1,1)).to(DEVICE)

writer.add_graph(model,sample_x.to(DEVICE))

for epoch in range(1000):
    loss = math.exp(-epoch/15)
    val_loss = 2 * math.exp(-epoch/15)
    writer.add_scalars(
        main_tag='loss',
        tag_scalar_dict={
            'training':loss,
            'validation': val_loss
        },
        global_step=epoch
    )

    writer.flush()

---

## Zadanie 2: Logowanie i wizualizacja postępu treningu

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split,KFold
from sklearn.preprocessing import StandardScaler

from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

import optuna
from tqdm import tqdm

c:\Users\posze\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [6]:
# physiochemical_protein
p_p = fetch_openml(data_id=42903, as_frame=True)

In [7]:
p_p.data.columns

Index(['RMSD', 'F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9'], dtype='object')

In [8]:
X = p_p.data[p_p.data.columns[1:]]
y = p_p.data.RMSD

In [9]:
X_trainval, X_test, y_trainval, y_test = train_test_split(X,y,test_size=0.3)

In [10]:
X_train, X_val, y_train, y_val = train_test_split(X_trainval,y_trainval,test_size=0.2)

In [11]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [12]:
print(X_test.shape)

(13719, 9)


In [13]:
class MLP(nn.Module):
    def __init__(self,n_hidden, hidden_dims):
        super().__init__()

        self.input = nn.Linear(9, hidden_dims[0])

        self.layers = nn.ModuleList([nn.Linear(hidden_dims[i],hidden_dims[i+1]) for i in range(n_hidden)])

        self.output = nn.Linear(hidden_dims[-1], 1)
        self.relu = nn.ReLU()

    def forward(self,X):
        X = self.relu(self.input(X))

        for layer in self.layers:
            X = self.relu(layer(X))
        
        return self.output(X)

In [14]:
N_EPOCH = 30

In [15]:
writer = SummaryWriter('runs/learning')
writer2 = SummaryWriter('runs/hiperparams')

In [30]:
def create_best_model(trainval_dataset):

    kfold = KFold(n_splits=5, shuffle=True)

    def objective(trial: optuna.Trial):
        batch_size = trial.suggest_int("BS", 32, 128, step=32)
        lr = trial.suggest_float("lr", 0.001, 0.1, log=True)
        n_hidden = trial.suggest_int("n_hidden",1,3, step=1)
        hidden_dims = []
        
        for i in range(n_hidden + 1):
            hidden_dim = trial.suggest_categorical(f"hidden_dim_{i}",[16,32,64,128])
            hidden_dims.append(hidden_dim)

        model = MLP(n_hidden, hidden_dims).to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.MSELoss()
        best_val_loss = float('inf')
        train_losses = []
        val_losses = []

        for fold, (train_idx, val_idx) in enumerate(kfold.split(trainval_dataset)):
            train_subset = torch.utils.data.Subset(trainval_dataset,train_idx)
            val_subset = torch.utils.data.Subset(trainval_dataset,val_idx)

            train_loader = DataLoader(train_subset, shuffle=True, batch_size=batch_size)
            val_loader = DataLoader(val_subset,shuffle=False, batch_size=batch_size)

            for epoch in range(N_EPOCH):
                model.train()
                train_loss = 0.0
                
                for X_train_batch, y_train_batch in train_loader:
                    X_train_batch,y_train_batch = X_train_batch.to(DEVICE),y_train_batch.to(DEVICE)



                    yhat = model(X_train_batch)

                    loss_batch = loss_fn(yhat,y_train_batch)
                    loss_batch.backward()

                    train_loss += loss_batch.item()

                    optimizer.step()
                    optimizer.zero_grad()
                
                train_loss /= len(train_loader)
                train_losses.append(train_loss)

                with torch.inference_mode():
                    
                    val_loss = 0.0
                    for X_val_batch, y_val_batch in val_loader:
                        X_val_batch, y_val_batch = X_val_batch.to(DEVICE), y_val_batch.to(DEVICE)

                        yhat = model(X_val_batch)

                        loss_batch = loss_fn(yhat, y_val_batch)
                        val_loss += loss_batch.item()

                    val_loss /= len(val_loader)
                    val_losses.append(val_loss)

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    
                    dic = {
                        'model_state': model.state_dict(),
                        'optimizer_state': optimizer.state_dict(),
                        'epoch': epoch,
                        'best_val_loss': best_val_loss,
                        'train_losses': train_losses,
                        'val_losses': val_losses,
                        'hyperparameters': {
                            'n_hidden': n_hidden,
                            'hidden_dims': hidden_dims,
                            'lr': lr,
                            'batch_size': batch_size
                        }
                    }

                    torch.save(dic, f'optuna_model.pth')

        writer.add_scalars(
            main_tag='training',
            tag_scalar_dict={
                'train_loss': train_loss,
                'val_loss': val_loss
            },
            global_step=trial.number
        )

        hparams = {
            'n_hidden': n_hidden,
            'lr': lr,
            'batch_size': batch_size
        }

        for i, hidden_dim in enumerate(hidden_dims):
            hparams[f'hidden_dim_{i}'] = hidden_dim

        metrics = {
            'val_loss': val_loss
        }

        writer2.add_hparams(
            hparams,
            metrics
        )

        writer2.flush()

        writer.flush()
        return val_loss
    
    study = optuna.study.create_study(direction='minimize')
    study.optimize(objective,n_trials=20)

    return study.best_params
        

In [31]:
from torch.utils.data import TensorDataset, ConcatDataset

trainval_dataset = ConcatDataset([
    TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train.to_numpy(), dtype=torch.float32).unsqueeze(1)
    ),
    TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val.to_numpy(), dtype=torch.float32).unsqueeze(1)
    )
])

best_hyperparams = create_best_model(trainval_dataset)
print(best_hyperparams)

[I 2026-04-16 23:26:30,973] A new study created in memory with name: no-name-01ec5a41-b013-47e5-a344-9d7e6a23ab43
[I 2026-04-16 23:29:22,301] Trial 0 finished with value: 13.763803297014379 and parameters: {'BS': 96, 'lr': 0.006525902901688351, 'n_hidden': 3, 'hidden_dim_0': 32, 'hidden_dim_1': 64, 'hidden_dim_2': 32, 'hidden_dim_3': 64}. Best is trial 0 with value: 13.763803297014379.
[I 2026-04-16 23:31:40,192] Trial 1 finished with value: 14.326003972221823 and parameters: {'BS': 128, 'lr': 0.013041719244462274, 'n_hidden': 3, 'hidden_dim_0': 128, 'hidden_dim_1': 32, 'hidden_dim_2': 32, 'hidden_dim_3': 16}. Best is trial 0 with value: 13.763803297014379.
[I 2026-04-16 23:38:01,694] Trial 2 finished with value: 13.722866611101141 and parameters: {'BS': 32, 'lr': 0.008161016512632306, 'n_hidden': 2, 'hidden_dim_0': 64, 'hidden_dim_1': 64, 'hidden_dim_2': 64}. Best is trial 2 with value: 13.722866611101141.
[I 2026-04-16 23:40:20,940] Trial 3 finished with value: 18.86592268587938 and 

{'BS': 96, 'lr': 0.004890472674742762, 'n_hidden': 2, 'hidden_dim_0': 64, 'hidden_dim_1': 128, 'hidden_dim_2': 128}


In [ ]:
best_model = MLP(best_hyperparams['n_hidden'], best_hyperparams['hidden_dims']).to(DEVICE)
checkpoint = torch.load('optuna_model.pth')
best_model.load_state_dict(checkpoint['model_state'])

test_dataset = torch.utils.data.TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test.to_numpy(), dtype=torch.float32).unsqueeze(1)
)

test_loader = DataLoader(test_dataset, batch_size=best_hyperparams['batch_size'], shuffle=False)

In [ ]:
best_model.eval()
test_loss = 0.0
loss_fn = nn.MSELoss()

with torch.inference_mode():
    for X_test_batch, y_test_batch in test_loader:
        X_test_batch, y_test_batch = X_test_batch.to(DEVICE), y_test_batch.to(DEVICE)

        yhat = best_model(X_test_batch)

        loss_batch = loss_fn(yhat, y_test_batch)
        test_loss += loss_batch.item()
    
test_loss /= len(test_loader)

print(f'Test loss: {test_loss}')

---